# Project: Image Captioning

---

In this notebook, you will learn how to load and pre-process data from the [COCO dataset](http://cocodataset.org/#home). You will also design a CNN-RNN model for automatically generating image captions.

Note that **any amendments that you make to this notebook will not be graded**.  However, you will use the instructions provided in **Step 3** and **Step 4** to implement your own CNN encoder and RNN decoder by making amendments to the **models.py** file provided as part of this project.  Your **models.py** file **will be graded**. 

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Explore the Data Loader
- [Step 2](#step2): Use the Data Loader to Obtain Batches
- [Step 3](#step3): Experiment with the CNN Encoder
- [Step 4](#step4): Implement the RNN Decoder

<a id='step1'></a>
## Step 1: Explore the Data Loader

We have already written a [data loader](http://pytorch.org/docs/master/data.html#torch.utils.data.DataLoader) that you can use to load the COCO dataset in batches. 

In the code cell below, you will initialize the data loader by using the `get_loader` function in **data_loader.py**.  

> For this project, you are not permitted to change the **data_loader.py** file, which must be used as-is.

The `get_loader` function takes as input a number of arguments that can be explored in **data_loader.py**.  Take the time to explore these arguments now by opening **data_loader.py** in a new window.  Most of the arguments must be left at their default values, and you are only allowed to amend the values of the arguments below:
1. **`transform`** - an [image transform](http://pytorch.org/docs/master/torchvision/transforms.html) specifying how to pre-process the images and convert them to PyTorch tensors before using them as input to the CNN encoder.  For now, you are encouraged to keep the transform as provided in `transform_train`.  You will have the opportunity later to choose your own image transform to pre-process the COCO images.
2. **`mode`** - one of `'train'` (loads the training data in batches) or `'test'` (for the test data). We will say that the data loader is in training or test mode, respectively.  While following the instructions in this notebook, please keep the data loader in training mode by setting `mode='train'`.
3. **`batch_size`** - determines the batch size.  When training the model, this is number of image-caption pairs used to amend the model weights in each training step.
4. **`vocab_threshold`** - the total number of times that a word must appear in the training captions before it is used as part of the vocabulary.  Words that have fewer than `vocab_threshold` occurrences in the training captions are considered unknown words. 
5. **`vocab_from_file`** - a Boolean that decides whether to load the vocabulary from file.  

We will describe the `vocab_threshold` and `vocab_from_file` arguments in more detail soon.  For now, run the code cell below.  Be patient - it may take a couple of minutes to run!

In [10]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import nltk
from torchvision import transforms


# Define project and dataset paths.
PROJECT_ROOT = Path.cwd().parent
COCO_ROOT = (Path.cwd() / "../../COCOapi").resolve()

# Allow imports from the project's src package.
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_loader

# Download NLTK tokenizer resources if needed.
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


# Define image preprocessing and augmentation for training.
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225),
    ),
])

# Vocabulary configuration.
vocab_threshold = 5

# Training batch size.
batch_size = 10

# Build the training data loader.
data_loader = get_loader(
    transform=transform_train,
    mode="train",
    batch_size=batch_size,
    vocab_threshold=vocab_threshold,
    vocab_from_file=False,
    cocoapi_loc=str(COCO_ROOT),
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
DEBUG cocoapi_loc: /Users/jimhuang/Udacity/Computer Vision/Practice/Course 4 Image Captioning/COCOapi
DEBUG img_folder: /Users/jimhuang/Udacity/Computer Vision/Practice/Course 4 Image Captioning/COCOapi/train2014
DEBUG annotations_file: /Users/jimhuang/Udacity/Computer Vision/Practice/Course 4 Image Captioning/COCOapi/annotations/captions_train2014.json
loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
[0/414113] Tokenizing captions...
[100000/414113] Tokenizing captions...
[200000/414113] Tokenizing captions...
[300000/414113] Tokenizing captions...
[400000/414113] Tokenizing captions...
Vocabulary successfully saved to ./vocab.pkl.
loading annotations into memory...
Done (t=0.21s)
creating index...
index created!
Obtaining caption lengths...


100%|██████████| 414113/414113 [00:08<00:00, 48373.72it/s]


In [12]:
print("Vocabulary size:", len(data_loader.dataset.vocab))

images, captions = next(iter(data_loader))

print("Images shape:", images.shape)
print("Captions shape:", captions.shape)

Vocabulary size: 8852
Images shape: torch.Size([10, 3, 224, 224])
Captions shape: torch.Size([10, 13])


<a id='step2'></a>
## Step 2: Verify Training Batches

The training data loader groups captions with the same sequence
length so that each batch can be represented as a dense tensor
without padding.

We verify the pipeline by loading one training batch and checking
the resulting tensor shapes.

In [13]:
# Load one training batch.
images, captions = next(iter(data_loader))

print("Image batch shape:", images.shape)
print("Caption batch shape:", captions.shape)

# Verify batch consistency.
assert images.size(0) == batch_size
assert captions.size(0) == batch_size
assert images.shape[1:] == (3, 224, 224)

print("Training batch validation passed.")

Image batch shape: torch.Size([10, 3, 224, 224])
Caption batch shape: torch.Size([10, 13])
Training batch validation passed.


<a id='step3'></a>
## Step 3: Experiment with the CNN Encoder

Run the code cell below to import `EncoderCNN` and `DecoderRNN` from **model.py**. 

In [21]:
# Watch for any changes in model.py, and re-load it automatically.
%load_ext autoreload
%autoreload 2

# Import EncoderCNN and DecoderRNN. 
from src.model import EncoderCNN, DecoderRNN

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In the next code cell we define a `device` that you will use move PyTorch tensors to GPU (if CUDA is available).  Run this code cell before continuing.

In [22]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


Run the code cell below to instantiate the CNN encoder in `encoder`.  

The pre-processed images from the batch in **Step 2** of this notebook are then passed through the encoder, and the output is stored in `features`.

In [23]:
# Specify the dimensionality of the image embedding.
embed_size = 256

#-#-#-# Do NOT modify the code below this line. #-#-#-#

# Initialize the encoder. (Optional: Add additional arguments if necessary.)
encoder = EncoderCNN(embed_size)

# Move the encoder to GPU if CUDA is available.
encoder.to(device)
    
# Move last batch of images (from Step 2) to GPU if CUDA is available.   
images = images.to(device)

# Pass the images through the encoder.
with torch.no_grad():
    features = encoder(images)

print("Image batch shape:", images.shape)
print("Feature shape:", features.shape)
print("Feature device:", features.device)

assert features.shape == (batch_size, embed_size)

print("CNN encoder validation passed.")

Image batch shape: torch.Size([10, 3, 224, 224])
Feature shape: torch.Size([10, 256])
Feature device: mps:0
CNN encoder validation passed.


The encoder that we provide to you uses the pre-trained ResNet-34 architecture (with the final fully-connected layer removed) to extract features from a batch of pre-processed images.  The output is then flattened to a vector, before being passed through a `Linear` layer to transform the feature vector to have the same size as the word embedding.

![Encoder](images/encoder.png)

You are welcome (and encouraged) to amend the encoder in **model.py**, to experiment with other architectures.  In particular, consider using a [different pre-trained model architecture](http://pytorch.org/docs/master/torchvision/models.html).  You may also like to [add batch normalization](http://pytorch.org/docs/master/nn.html#normalization-layers).  

> You are **not** required to change anything about the encoder.

For this project, you **must** incorporate a pre-trained CNN into your encoder.  Your `EncoderCNN` class must take `embed_size` as an input argument, which will also correspond to the dimensionality of the input to the RNN decoder that you will implement in Step 4.  When you train your model in the next notebook in this sequence (**2_Training.ipynb**), you are welcome to tweak the value of `embed_size`.

If you decide to modify the `EncoderCNN` class, save **model.py** and re-execute the code cell above.  If the code cell returns an assertion error, then please follow the instructions to modify your code before proceeding.  The assert statements ensure that `features` is a PyTorch tensor with shape `[batch_size, embed_size]`.

<a id='step4'></a>
## Step 4: Implement the RNN Decoder

The decoder combines encoded image features with caption token embeddings

and uses an LSTM to predict vocabulary logits at each sequence position.

For a batch of size `B`, caption length `T`, and vocabulary size `V`,

the decoder output has shape `[B, T, V]`.

In [25]:
# Decoder configuration.
hidden_size = 512
vocab_size = len(data_loader.dataset.vocab)

# Initialize the decoder.
decoder = DecoderRNN(
    embed_size=embed_size,
    hidden_size=hidden_size,
    vocab_size=vocab_size,
).to(device)

# Move captions to the same device.
captions = captions.to(device)

# Validate the decoder forward pass.
with torch.no_grad():
    outputs = decoder(features, captions)

print("Decoder output shape:", outputs.shape)
print("Decoder output device:", outputs.device)

assert isinstance(outputs, torch.Tensor)
assert outputs.shape == (
    batch_size,
    captions.shape[1],
    vocab_size,
)

print("RNN decoder validation passed.")

Decoder output shape: torch.Size([10, 13, 8852])
Decoder output device: mps:0
RNN decoder validation passed.


When you train your model in the next notebook in this sequence (**2_Training.ipynb**), you are welcome to tweak the value of `hidden_size`.